In [1]:
import warnings
warnings.filterwarnings("ignore")

import math
import random
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [2]:
ratings = pd.read_csv(
    "../data/u.data",
    sep="\t",
    names=[
        "user_id",
        "movie_id",
        "rating",
        "timestamp"
    ]
)

movies = pd.read_csv(
    "../data/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

In [3]:
genre_names = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "Film-Noir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "Sci-Fi",
    "Thriller",
    "War",
    "Western"
]

In [4]:
ratings = ratings.sort_values(
    ["user_id", "timestamp"]
).reset_index(drop=True)

In [5]:
user_mapping = {
    user: idx
    for idx, user in enumerate(
        ratings.user_id.unique()
    )
}

movie_mapping = {
    movie: idx + 1
    for idx, movie in enumerate(
        ratings.movie_id.unique()
    )
}

ratings["user_id"] = ratings.user_id.map(user_mapping)
ratings["movie_id"] = ratings.movie_id.map(movie_mapping)

In [6]:
PAD_TOKEN = 0

NUM_USERS = ratings.user_id.nunique()

NUM_MOVIES = ratings.movie_id.nunique()

MASK_TOKEN = NUM_MOVIES + 1

NUM_ITEMS = NUM_MOVIES + 2

NUM_ATTRIBUTES = len(genre_names)

print("Users      :", NUM_USERS)
print("Movies     :", NUM_MOVIES)
print("MASK TOKEN :", MASK_TOKEN)
print("NUM ITEMS  :", NUM_ITEMS)

Users      : 943
Movies     : 1682
MASK TOKEN : 1683
NUM ITEMS  : 1684


In [7]:
user_sequences = (
    ratings
    .groupby("user_id")["movie_id"]
    .apply(list)
)

In [8]:
item_attributes = {}

for _, row in movies.iterrows():

    original_movie = row[0]

    if original_movie not in movie_mapping:
        continue

    mapped_movie = movie_mapping[original_movie]

    attributes = []

    for i in range(NUM_ATTRIBUTES):

        if row[5 + i] == 1:
            attributes.append(i + 1)

    item_attributes[mapped_movie] = attributes

In [9]:
movie = random.choice(
    list(item_attributes.keys())
)

print(movie)

print(item_attributes[movie])

print()

for attribute in item_attributes[movie]:
    print(genre_names[attribute - 1])

1308
[9]

Drama


In [10]:
MAX_LEN = 50

pretrain_sequences = []

train_sequences = []
train_targets = []

valid_sequences = []
valid_targets = []

test_sequences = []
test_targets = []

for sequence in user_sequences:

    if len(sequence) < 5:
        continue

    # ---------- Pretraining ----------
    pretrain_sequences.append(sequence[:-2])

    # ---------- Fine-tuning ----------
    train_sequences.append(sequence[:-2])
    train_targets.append(sequence[-2])

    valid_sequences.append(sequence[:-1])
    valid_targets.append(sequence[-1])

    test_sequences.append(sequence[:-1])
    test_targets.append(sequence[-1])

In [11]:
def left_pad(sequence, max_len):

    sequence = sequence[-max_len:]

    padding = [PAD_TOKEN] * (max_len - len(sequence))

    return padding + sequence

In [12]:
def create_aap_labels(
    sequence,
    masked_positions
):

    labels = np.zeros(
        (
            len(sequence),
            NUM_ATTRIBUTES
        ),
        dtype=np.float32
    )

    masked_positions = set(masked_positions)

    for i, item in enumerate(sequence):

        if i in masked_positions:
            continue

        if item not in item_attributes:
            continue

        for attribute in item_attributes[item]:

            labels[i, attribute - 1] = 1

    return labels

In [13]:
def create_map_labels(
    sequence,
    masked_positions
):

    positive_attributes = np.zeros(
        len(sequence),
        dtype=np.int64
    )

    negative_attributes = np.zeros(
        len(sequence),
        dtype=np.int64
    )

    for position in masked_positions:

        item = sequence[position]

        attributes = item_attributes[item]

        positive = random.choice(attributes)

        while True:

            negative = random.randint(
                1,
                NUM_ATTRIBUTES
            )

            if negative not in attributes:
                break

        positive_attributes[position] = positive

        negative_attributes[position] = negative

    return (
        positive_attributes,
        negative_attributes
    )

In [14]:
def create_segment_prediction(sequence):

    sequence = sequence.copy()

    length = len(sequence)

    segment_length = random.randint(
        1,
        max(1, length // 2)
    )

    start = random.randint(
        0,
        length - segment_length
    )

    positive_segment = sequence[
        start:
        start + segment_length
    ]

    while True:

        random_sequence = random.choice(
            pretrain_sequences
        )

        if len(random_sequence) >= segment_length:
            break

    random_start = random.randint(
        0,
        len(random_sequence) - segment_length
    )

    negative_segment = random_sequence[
        random_start:
        random_start + segment_length
    ]

    context_sequence = sequence.copy()

    context_sequence[
        start:
        start + segment_length
    ] = [MASK_TOKEN] * segment_length

    return (
        context_sequence,
        positive_segment,
        negative_segment
    )

In [15]:
def mask_items(
    sequence,
    mask_prob=0.2,
    mask_token=MASK_TOKEN,
    pad_token=PAD_TOKEN
):

    """
    BERT-style masked item modeling for S3Rec.

    For each chosen position we record:
      * the original item   -> positive_items
      * a sampled distractor-> negative_items (never the original,
                              never the [MASK] token)
      * the position itself -> masked_positions

    The distractor is sampled in [1, NUM_MOVIES] (which excludes
    MASK_TOKEN, since MASK_TOKEN = NUM_MOVIES + 1).

    To guarantee MIP / MAP have a non-empty target for every sample,
    we enforce that at least one valid position is masked.
    """

    sequence = list(sequence)

    length = len(sequence)

    masked_sequence = list(sequence)

    pos_items = np.zeros(length, dtype=np.int64)

    neg_items = np.zeros(length, dtype=np.int64)

    masked_positions = []

    for i in range(length):

        if sequence[i] == pad_token:
            continue

        if random.random() < mask_prob:

            masked_positions.append(i)

            pos_items[i] = sequence[i]

            # ---- Negative sampling for MIP ----
            while True:

                neg = random.randint(1, NUM_MOVIES)

                if neg != sequence[i]:
                    break

            neg_items[i] = neg

            r = random.random()

            if r < 0.8:

                masked_sequence[i] = mask_token

            elif r < 0.9:

                # ---- Random replacement (10% branch combined) ----
                # Must differ from the original item and from MASK_TOKEN.
                while True:

                    rand_item = random.randint(1, NUM_MOVIES)

                    if rand_item != sequence[i]:
                        break

                masked_sequence[i] = rand_item

            # else: keep original (10%)

    # ---- Guarantee at least one masked position ----
    if len(masked_positions) == 0:

        # Pick a random non-pad position.
        while True:

            pos = random.randrange(length)

            if sequence[pos] != pad_token:
                break

        masked_positions.append(pos)

        pos_items[pos] = sequence[pos]

        while True:

            neg = random.randint(1, NUM_MOVIES)

            if neg != sequence[pos]:
                break

        neg_items[pos] = neg

        masked_sequence[pos] = mask_token

    return (
        masked_sequence,
        pos_items,
        neg_items,
        masked_positions
    )

In [16]:
sequence = pretrain_sequences[0]

masked_sequence, pos_items, neg_items, masked_positions = mask_items(sequence)

print(masked_sequence)

print(masked_positions)

print(pos_items)

print(neg_items)

aap = create_aap_labels(
    sequence,
    masked_positions
)

print(aap.shape)

map_pos, map_neg = create_map_labels(
    sequence,
    masked_positions
)

print(map_pos)

print(map_neg)

context, positive, negative = create_segment_prediction(sequence)

print(context)

print(positive)

print(negative)

[1683, 2, 3, 4, 5, 1683, 1683, 1683, 9, 10, 11, 12, 13, 14, 1683, 1683, 1683, 18, 19, 1683, 21, 22, 23, 24, 25, 26, 27, 28, 1683, 30, 31, 32, 33, 34, 35, 1683, 37, 38, 39, 40, 1683, 42, 43, 44, 45, 46, 47, 48, 822, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 314, 70, 71, 72, 1683, 74, 75, 76, 77, 78, 1683, 80, 81, 82, 83, 84, 85, 86, 612, 88, 89, 1683, 91, 92, 93, 1683, 1653, 96, 97, 1558, 99, 1683, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 1683, 1683, 116, 1683, 146, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 1683, 130, 131, 132, 133, 134, 1683, 136, 1683, 138, 139, 140, 141, 142, 143, 144, 145, 192, 147, 148, 1683, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 1683, 1683, 163, 1683, 165, 166, 1683, 1683, 169, 170, 171, 172, 173, 1683, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 1683, 187, 1683, 189, 190, 191, 192, 193, 1683, 195, 196, 197, 1683, 199, 1683, 201, 202, 1683, 204, 205, 206, 207, 208, 209, 2

In [17]:
class S3RecPretrainDataset(Dataset):

    def __init__(self, sequences):

        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):

        sequence = self.sequences[idx]

        (
            masked_sequence,
            positive_items,
            negative_items,
            masked_positions
        ) = mask_items(sequence)

        aap_labels = create_aap_labels(
            sequence,
            masked_positions
        )

        # Truncate to match left_pad convention (keeps the tail).
        if len(aap_labels) > MAX_LEN:
            aap_labels = aap_labels[-MAX_LEN:]

        (
            positive_attributes,
            negative_attributes
        ) = create_map_labels(
            sequence,
            masked_positions
        )

        (
            context_sequence,
            positive_segment,
            negative_segment
        ) = create_segment_prediction(sequence)

        return {

            "masked_sequence":
                torch.LongTensor(
                    left_pad(masked_sequence, MAX_LEN)
                ),

            "positive_items":
                torch.LongTensor(
                    left_pad(
                        positive_items.tolist(),
                        MAX_LEN
                    )
                ),

            "negative_items":
                torch.LongTensor(
                    left_pad(
                        negative_items.tolist(),
                        MAX_LEN
                    )
                ),

            "aap_labels":
                torch.FloatTensor(
                    np.vstack([
                        np.zeros((MAX_LEN-len(aap_labels), NUM_ATTRIBUTES)),
                        aap_labels
                    ])
                ),

            "positive_attributes":
                torch.LongTensor(
                    left_pad(
                        positive_attributes.tolist(),
                        MAX_LEN
                    )
                ),

            "negative_attributes":
                torch.LongTensor(
                    left_pad(
                        negative_attributes.tolist(),
                        MAX_LEN
                    )
                ),

            "context_sequence":
                torch.LongTensor(
                    left_pad(
                        context_sequence,
                        MAX_LEN
                    )
                ),

            "positive_segment":
                torch.LongTensor(
                    left_pad(
                        positive_segment,
                        MAX_LEN
                    )
                ),

            "negative_segment":
                torch.LongTensor(
                    left_pad(
                        negative_segment,
                        MAX_LEN
                    )
                )
        }

In [18]:
pretrain_dataset = S3RecPretrainDataset(
    pretrain_sequences
)

pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=64,
    shuffle=True
)

In [19]:
batch = next(iter(pretrain_loader))

for k, v in batch.items():
    print(k, v.shape)

masked_sequence torch.Size([64, 50])
positive_items torch.Size([64, 50])
negative_items torch.Size([64, 50])
aap_labels torch.Size([64, 50, 19])
positive_attributes torch.Size([64, 50])
negative_attributes torch.Size([64, 50])
context_sequence torch.Size([64, 50])
positive_segment torch.Size([64, 50])
negative_segment torch.Size([64, 50])


In [20]:
class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, hidden_size):

        super().__init__()

        self.embedding = nn.Embedding(
            max_len,
            hidden_size
        )

    def forward(self, x):

        batch_size, seq_len = x.size()

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0)

        return self.embedding(positions)

In [21]:
class TransformerEncoder(nn.Module):

    def __init__(
        self,
        hidden_size=64,
        num_heads=2,
        num_layers=2,
        dropout=0.2
    ):

        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=hidden_size,

            nhead=num_heads,

            dim_feedforward=hidden_size * 4,

            dropout=dropout,

            activation="gelu",

            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(

            encoder_layer,

            num_layers=num_layers
        )

    def forward(
        self,
        x,
        padding_mask
    ):

        return self.encoder(

            x,

            src_key_padding_mask=padding_mask
        )

In [22]:
class S3RecBackbone(nn.Module):

    def __init__(

        self,

        num_items,

        hidden_size=64,

        max_len=50,

        num_heads=2,

        num_layers=2,

        dropout=0.2

    ):

        super().__init__()

        self.item_embedding = nn.Embedding(

            num_items,

            hidden_size,

            padding_idx=PAD_TOKEN

        )

        self.position_embedding = PositionalEmbedding(

            max_len,

            hidden_size

        )

        self.dropout = nn.Dropout(dropout)

        self.layer_norm = nn.LayerNorm(hidden_size)

        self.encoder = TransformerEncoder(

            hidden_size,

            num_heads,

            num_layers,

            dropout

        )

    def forward(
        self,
        sequence
    ):

        positions = self.position_embedding(sequence)

        items = self.item_embedding(sequence)

        x = items + positions

        x = self.layer_norm(x)

        x = self.dropout(x)

        padding_mask = sequence.eq(PAD_TOKEN)

        output = self.encoder(

            x,

            padding_mask

        )

        return output

In [23]:
backbone = S3RecBackbone(

    num_items=NUM_ITEMS,

    hidden_size=64,

    max_len=MAX_LEN

)

batch = next(iter(pretrain_loader))

output = backbone(

    batch["masked_sequence"]

)

print(output.shape)

torch.Size([64, 50, 64])


In [24]:
class AAPHead(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_attributes
    ):

        super().__init__()

        self.classifier = nn.Linear(
            hidden_size,
            num_attributes
        )

    def forward(self, hidden):

        return self.classifier(hidden)
    

In [25]:
class MIPHead(nn.Module):

    def __init__(
        self,
        hidden_size,
        item_embedding
    ):

        super().__init__()

        self.item_embedding = item_embedding

    def forward(

        self,

        hidden,

        positive_items,

        negative_items

    ):

        positive_embedding = self.item_embedding(
            positive_items
        )

        negative_embedding = self.item_embedding(
            negative_items
        )

        positive_score = (
            hidden * positive_embedding
        ).sum(-1)

        negative_score = (
            hidden * negative_embedding
        ).sum(-1)

        return positive_score, negative_score

In [26]:
class MAPHead(nn.Module):

    def __init__(

        self,

        hidden_size,

        num_attributes

    ):

        super().__init__()

        self.attribute_embedding = nn.Embedding(

            num_attributes + 1,

            hidden_size

        )

    def forward(

        self,

        hidden,

        positive_attributes,

        negative_attributes

    ):

        positive_embedding = self.attribute_embedding(

            positive_attributes

        )

        negative_embedding = self.attribute_embedding(

            negative_attributes

        )

        positive_score = (

            hidden * positive_embedding

        ).sum(-1)

        negative_score = (

            hidden * negative_embedding

        ).sum(-1)

        return positive_score, negative_score

In [27]:
class SPHead(nn.Module):

    """
    Segment Prediction head.

    Pools a context sequence and a candidate segment into fixed
    vectors, then scores how well the context predicts the segment.

    Both the context and the candidate segment use *masked* mean
    pooling over non-pad tokens. The plain .mean(1) used in early
    drafts would dilute the context signal with PAD embeddings.
    """

    def __init__(

        self,

        hidden_size,

        item_embedding

    ):

        super().__init__()

        self.item_embedding = item_embedding

    @staticmethod
    def masked_mean(hidden, pad_mask):

        """
        hidden:   (B, L, H)
        pad_mask: (B, L)  — True where the token is PAD.

        Returns (B, H) — the mean over non-pad positions.
        """

        keep = (~pad_mask).float().unsqueeze(-1)             # (B, L, 1)

        summed = (hidden * keep).sum(dim=1)                  # (B, H)

        denom = keep.sum(dim=1).clamp(min=1.0)               # (B, 1)

        return summed / denom

    def encode_segment(

        self,

        segment

    ):

        embedding = self.item_embedding(segment)

        pad_mask = segment.eq(PAD_TOKEN)

        return self.masked_mean(embedding, pad_mask)

    def forward(

        self,

        context_hidden,

        context_sequence,

        positive_segment,

        negative_segment

    ):

        # Pool the context with a PAD-aware mean.
        context_pad_mask = context_sequence.eq(PAD_TOKEN)

        context_vector = self.masked_mean(
            context_hidden,
            context_pad_mask
        )

        positive_vector = self.encode_segment(positive_segment)

        negative_vector = self.encode_segment(negative_segment)

        positive_score = (

            context_vector * positive_vector

        ).sum(-1)

        negative_score = (

            context_vector * negative_vector

        ).sum(-1)

        return positive_score, negative_score

In [28]:
class S3Rec(nn.Module):

    def __init__(

        self,

        num_items,

        num_attributes,

        hidden_size=64,

        max_len=50,

        num_heads=2,

        num_layers=2,

        dropout=0.2

    ):

        super().__init__()

        self.backbone = S3RecBackbone(

            num_items=num_items,

            hidden_size=hidden_size,

            max_len=max_len,

            num_heads=num_heads,

            num_layers=num_layers,

            dropout=dropout

        )

        self.aap_head = AAPHead(

            hidden_size,

            num_attributes

        )

        self.mip_head = MIPHead(

            hidden_size,

            self.backbone.item_embedding

        )

        self.map_head = MAPHead(

            hidden_size,

            num_attributes

        )

        self.sp_head = SPHead(

            hidden_size,

            self.backbone.item_embedding

        )

    def forward(

        self,

        batch

    ):

        hidden = self.backbone(

            batch["masked_sequence"]

        )

        aap_logits = self.aap_head(

            hidden

        )

        mip_pos, mip_neg = self.mip_head(

            hidden,

            batch["positive_items"],

            batch["negative_items"]

        )

        map_pos, map_neg = self.map_head(

            hidden,

            batch["positive_attributes"],

            batch["negative_attributes"]

        )

        context_hidden = self.backbone(

            batch["context_sequence"]

        )

        sp_pos, sp_neg = self.sp_head(

            context_hidden,

            batch["context_sequence"],

            batch["positive_segment"],

            batch["negative_segment"]

        )

        return {

            "aap_logits": aap_logits,

            "mip_pos": mip_pos,

            "mip_neg": mip_neg,

            "map_pos": map_pos,

            "map_neg": map_neg,

            "sp_pos": sp_pos,

            "sp_neg": sp_neg

        }

In [29]:
model = S3Rec(

    num_items=NUM_ITEMS,

    num_attributes=NUM_ATTRIBUTES,

    hidden_size=64,

    max_len=MAX_LEN

)

batch = next(iter(pretrain_loader))

outputs = model(batch)

for key, value in outputs.items():
    print(key, value.shape)

aap_logits torch.Size([64, 50, 19])
mip_pos torch.Size([64, 50])
mip_neg torch.Size([64, 50])
map_pos torch.Size([64, 50])
map_neg torch.Size([64, 50])
sp_pos torch.Size([64])
sp_neg torch.Size([64])


In [30]:
class S3RecPretrainLoss(nn.Module):

    """
    Multi-task loss with proper position-wise masking.

    - AAP:  ignore padded positions (masked positions already have zeroed
           labels in the dataset).
    - MIP:  only masked positions matter (positive_items is non-zero
           only at masked positions).
    - MAP:  same convention as MIP.
    - SP:   per-sample BCE; segments are guaranteed non-empty by the
           upstream filter on pretrain_sequences.
    """

    def __init__(self):

        super().__init__()

        # reduction='none' so we can apply position-wise validity masks
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, outputs, batch):

        #################################################
        # AAP: ignore padded positions
        #################################################

        aap_logits = outputs['aap_logits']                       # (B, L, A)
        aap_labels = batch['aap_labels']                         # (B, L, A)

        pad_mask = batch['masked_sequence'].eq(PAD_TOKEN)        # (B, L)
        aap_valid = (~pad_mask).float().unsqueeze(-1)            # (B, L, 1)

        aap_loss = self.bce(aap_logits, aap_labels)
        aap_loss = (aap_loss * aap_valid).sum() / aap_valid.sum().clamp(min=1.0)

        #################################################
        # MIP: only masked positions
        #################################################

        mip_mask = batch['positive_items'].ne(PAD_TOKEN).float() # (B, L)

        mip_pos = self.bce(outputs['mip_pos'], torch.ones_like(outputs['mip_pos']))
        mip_neg = self.bce(outputs['mip_neg'], torch.zeros_like(outputs['mip_neg']))

        mip_loss = ((mip_pos + mip_neg) * mip_mask).sum() / mip_mask.sum().clamp(min=1.0)

        #################################################
        # MAP: only masked positions
        #################################################

        map_mask = batch['positive_attributes'].ne(PAD_TOKEN).float()

        map_pos = self.bce(outputs['map_pos'], torch.ones_like(outputs['map_pos']))
        map_neg = self.bce(outputs['map_neg'], torch.zeros_like(outputs['map_neg']))

        map_loss = ((map_pos + map_neg) * map_mask).sum() / map_mask.sum().clamp(min=1.0)

        #################################################
        # SP: per-sample BCE
        #################################################

        sp_pos = self.bce(outputs['sp_pos'], torch.ones_like(outputs['sp_pos']))
        sp_neg = self.bce(outputs['sp_neg'], torch.zeros_like(outputs['sp_neg']))

        sp_loss = (sp_pos + sp_neg).mean()

        #################################################

        total = aap_loss + mip_loss + map_loss + sp_loss

        return {

            'loss': total,
            'aap':  aap_loss.detach(),
            'mip':  mip_loss.detach(),
            'map':  map_loss.detach(),
            'sp':   sp_loss.detach()
        }

In [31]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(device)

cpu


In [32]:
model = S3Rec(

    num_items=NUM_ITEMS,

    num_attributes=NUM_ATTRIBUTES,

    hidden_size=64,

    max_len=MAX_LEN,

    num_heads=2,

    num_layers=2

).to(device)

In [33]:
criterion = S3RecPretrainLoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=1e-3,

    weight_decay=1e-5

)

In [34]:
EPOCHS = 20

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    running_aap = 0

    running_mip = 0

    running_map = 0

    running_sp = 0

    for batch in pretrain_loader:

        for key in batch:

            batch[key] = batch[key].to(device)

        optimizer.zero_grad()

        outputs = model(batch)

        losses = criterion(

            outputs,

            batch

        )

        losses["loss"].backward()

        optimizer.step()

        running_loss += losses["loss"].item()

        running_aap += losses["aap"].item()

        running_mip += losses["mip"].item()

        running_map += losses["map"].item()

        running_sp += losses["sp"].item()

    n = len(pretrain_loader)

    print(

        f"Epoch {epoch+1:02d}",

        f"Loss={running_loss/n:.4f}",

        f"AAP={running_aap/n:.4f}",

        f"MIP={running_mip/n:.4f}",

        f"MAP={running_map/n:.4f}",

        f"SP={running_sp/n:.4f}"

    )

Epoch 01 Loss=22.5629 AAP=10.5621 MIP=6.0152 MAP=4.2332 SP=1.7524
Epoch 02 Loss=16.6204 AAP=6.3578 MIP=5.6575 MAP=2.4323 SP=2.1727
Epoch 03 Loss=14.9173 AAP=5.4090 MIP=5.3468 MAP=1.9870 SP=2.1744
Epoch 04 Loss=13.9432 AAP=5.1439 MIP=4.9480 MAP=1.8188 SP=2.0325
Epoch 05 Loss=13.2368 AAP=5.0240 MIP=4.5903 MAP=1.7183 SP=1.9043
Epoch 06 Loss=12.7072 AAP=4.9914 MIP=4.1895 MAP=1.6148 SP=1.9115
Epoch 07 Loss=12.2989 AAP=4.9508 MIP=3.9949 MAP=1.5045 SP=1.8486
Epoch 08 Loss=11.9643 AAP=4.9271 MIP=3.6740 MAP=1.5049 SP=1.8582
Epoch 09 Loss=11.6076 AAP=4.9160 MIP=3.5058 MAP=1.4556 SP=1.7301
Epoch 10 Loss=11.3297 AAP=4.8941 MIP=3.2801 MAP=1.4003 SP=1.7551
Epoch 11 Loss=11.0150 AAP=4.8876 MIP=3.0893 MAP=1.3504 SP=1.6876
Epoch 12 Loss=10.6961 AAP=4.8785 MIP=2.8817 MAP=1.3365 SP=1.5994
Epoch 13 Loss=10.5722 AAP=4.8753 MIP=2.7775 MAP=1.2874 SP=1.6320
Epoch 14 Loss=10.4040 AAP=4.8652 MIP=2.6760 MAP=1.2827 SP=1.5801
Epoch 15 Loss=10.1925 AAP=4.8712 MIP=2.5006 MAP=1.2415 SP=1.5792
Epoch 16 Loss=10.0579 AA

In [35]:
torch.save(

    model.state_dict(),

    "s3rec_pretrained.pth"

)

print("Pretraining completed.")

Pretraining completed.


In [36]:
class S3RecFineTuneDataset(Dataset):

    def __init__(self, sequences, targets):

        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):

        sequence = left_pad(
            self.sequences[idx],
            MAX_LEN
        )

        target = self.targets[idx]

        return {

            "sequence": torch.LongTensor(sequence),

            "target": torch.LongTensor([target])

        }

In [37]:
train_dataset = S3RecFineTuneDataset(
    train_sequences,
    train_targets
)

valid_dataset = S3RecFineTuneDataset(
    valid_sequences,
    valid_targets
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

In [38]:
class CausalTransformerEncoder(nn.Module):

    """
    Identical internal architecture to TransformerEncoder, but adds an
    upper-triangular causal mask so position t can only attend to
    positions <= t. Used during fine-tuning, where the model must not
    see future items when predicting the next one.
    """

    def __init__(
        self,
        hidden_size=64,
        num_heads=2,
        num_layers=2,
        dropout=0.2
    ):

        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(

            d_model=hidden_size,

            nhead=num_heads,

            dim_feedforward=hidden_size * 4,

            dropout=dropout,

            activation="gelu",

            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(

            encoder_layer,

            num_layers=num_layers
        )

    def forward(self, x, padding_mask):

        seq_len = x.size(1)

        # True above the diagonal: position i cannot attend to position j > i.
        causal_mask = torch.triu(

            torch.ones(

                seq_len,
                seq_len,
                device=x.device,
                dtype=torch.bool

            ),

            diagonal=1
        )

        return self.encoder(

            x,

            mask=causal_mask,

            src_key_padding_mask=padding_mask
        )


class S3RecFinetuneModel(nn.Module):

    """
    Fine-tuning model that reuses the pretrained S3RecBackbone.

    Differences from the pretraining backbone:
      * causal attention (no peeking at future items)
      * hidden state is taken at the LAST NON-PADDING position,
        because sequences are left-padded — hidden[:, -1] would land
        on the pad token for any sequence shorter than MAX_LEN.

    Item logits come from the tied item embedding matrix (output layer
    = item_embedding.weight.T), which is a standard BERT4Rec trick.
    """

    def __init__(
        self,
        pretrained_backbone,
        hidden_size=64,
        max_len=50,
        num_heads=2,
        num_layers=2,
        dropout=0.2
    ):

        super().__init__()

        # Same components as S3RecBackbone, but with a causal encoder.
        self.item_embedding = nn.Embedding(

            pretrained_backbone.item_embedding.num_embeddings,
            hidden_size,
            padding_idx=PAD_TOKEN
        )

        self.position_embedding = PositionalEmbedding(max_len, hidden_size)

        self.dropout = nn.Dropout(dropout)

        self.layer_norm = nn.LayerNorm(hidden_size)

        self.encoder = CausalTransformerEncoder(

            hidden_size,
            num_heads,
            num_layers,
            dropout
        )

        self._load_pretrained_backbone(pretrained_backbone)

    def _load_pretrained_backbone(self, pretrained_backbone):

        self.item_embedding.load_state_dict(
            pretrained_backbone.item_embedding.state_dict()
        )

        self.position_embedding.load_state_dict(
            pretrained_backbone.position_embedding.state_dict()
        )

        self.layer_norm.load_state_dict(
            pretrained_backbone.layer_norm.state_dict()
        )

        # The encoder layer architecture is identical between
        # TransformerEncoder and CausalTransformerEncoder, so the
        # state dicts are compatible — the causal mask is applied at
        # forward time, not stored in the parameters.
        self.encoder.load_state_dict(
            pretrained_backbone.encoder.state_dict()
        )

    def forward(self, sequence):

        positions = self.position_embedding(sequence)

        items = self.item_embedding(sequence)

        x = items + positions

        x = self.layer_norm(x)

        x = self.dropout(x)

        padding_mask = sequence.eq(PAD_TOKEN)

        hidden = self.encoder(x, padding_mask)

        return hidden

    def get_last_hidden(self, sequence, hidden):

        """
        Gather the hidden state at the last non-padding position.

        hidden: (B, L, H)
        sequence: (B, L)  — 0 at padded positions
        """

        # Number of valid (non-pad) tokens per row.
        seq_lens = sequence.ne(PAD_TOKEN).long().sum(dim=1)         # (B,)
        last_idx = (seq_lens - 1).clamp(min=0)                      # (B,)

        batch_idx = torch.arange(

            sequence.size(0),
            device=sequence.device

        )

        return hidden[batch_idx, last_idx]                          # (B, H)

    def predict(self, sequence):

        hidden = self.forward(sequence)

        last_hidden = self.get_last_hidden(sequence, hidden)        # (B, H)

        # Tied projection: (B, H) @ (H, V) -> (B, V)
        logits = last_hidden @ self.item_embedding.weight.T

        return logits

In [39]:
finetune_model = S3RecFinetuneModel(
    model.backbone
).to(device)

print(finetune_model)

S3RecFinetuneModel(
  (item_embedding): Embedding(1684, 64, padding_idx=0)
  (position_embedding): PositionalEmbedding(
    (embedding): Embedding(50, 64)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (encoder): CausalTransformerEncoder(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
          (dropout1): Dropout(p=0.2, inplace=False)
  

In [40]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    finetune_model.parameters(),

    lr=1e-4

)

In [41]:
EPOCHS = 30

for epoch in range(EPOCHS):

    finetune_model.train()

    total_loss = 0

    for batch in train_loader:

        sequence = batch["sequence"].to(device)

        target = batch["target"].squeeze().to(device)

        optimizer.zero_grad()

        # Use .predict() — forward() returns hidden states only.
        logits = finetune_model.predict(sequence)

        loss = criterion(

            logits,

            target

        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(

        f"Epoch {epoch+1:02d}",

        f"Loss {total_loss/len(train_loader):.4f}"

    )

Epoch 01 Loss 15.9486
Epoch 02 Loss 14.5459
Epoch 03 Loss 13.6525
Epoch 04 Loss 13.0499
Epoch 05 Loss 12.6415
Epoch 06 Loss 12.4227
Epoch 07 Loss 12.1535
Epoch 08 Loss 11.7870
Epoch 09 Loss 11.7074
Epoch 10 Loss 11.4193
Epoch 11 Loss 11.2391
Epoch 12 Loss 11.1485
Epoch 13 Loss 10.9847
Epoch 14 Loss 10.9141
Epoch 15 Loss 10.7616
Epoch 16 Loss 10.7712
Epoch 17 Loss 10.6016
Epoch 18 Loss 10.5043
Epoch 19 Loss 10.4219
Epoch 20 Loss 10.3572
Epoch 21 Loss 10.3369
Epoch 22 Loss 10.1025
Epoch 23 Loss 10.1313
Epoch 24 Loss 9.9899
Epoch 25 Loss 9.9097
Epoch 26 Loss 9.9478
Epoch 27 Loss 9.8239
Epoch 28 Loss 9.7578
Epoch 29 Loss 9.7948
Epoch 30 Loss 9.6295


In [42]:
def recommend(
    model,
    sequence,
    k=10
):

    """
    Top-K recommendation for a single sequence. Items already in the
    input sequence are masked out so we never recommend something the
    user has already interacted with.
    """

    model.eval()

    with torch.no_grad():

        padded = torch.LongTensor(

            left_pad(sequence, MAX_LEN)

        ).unsqueeze(0).to(device)

        logits = model.predict(padded)

        # Mask out everything in the input sequence (incl. PAD, MASK).
        seen = padded.unique()
        logits[0, seen] = float('-inf')

        _, indices = torch.topk(

            logits.squeeze(),
            k
        )

    return indices.cpu().numpy()

In [43]:
recommend(
    finetune_model,
    train_sequences[0],
    10
)

array([1310, 1683,  580,  735, 1325, 1002, 1235,  681,  437, 1660])

In [44]:
def hit_rate(model, loader, k=10):

    """
    HR@K: fraction of users whose ground-truth next item appears in
    the top-K predictions. Items already seen are excluded.
    """

    model.eval()

    hits = 0

    total = 0

    with torch.no_grad():

        for batch in loader:

            sequence = batch["sequence"].to(device)

            target = batch["target"].squeeze().to(device)

            logits = model.predict(sequence)

            # Mask out items seen in the input (incl. PAD / MASK).
            seen_mask = torch.zeros_like(logits, dtype=torch.bool)
            seen_mask.scatter_(1, sequence, True)

            seen_mask[:, PAD_TOKEN]  = True
            seen_mask[:, MASK_TOKEN] = True

            logits = logits.masked_fill(seen_mask, float('-inf'))

            topk = torch.topk(

                logits,
                k,
                dim=1

            ).indices

            hits += (

                topk == target.unsqueeze(1)

            ).any(dim=1).sum().item()

            total += target.size(0)

    return hits / total

In [45]:
def ndcg(model, loader, k=10):

    """
    NDCG@K (binary relevance, single relevant item):
        DCG  = 1 / log2(rank + 2)  if rank in [0, K), else 0
        IDCG = 1 / log2(0 + 2) = 1
    """

    model.eval()

    score = 0

    total = 0

    with torch.no_grad():

        for batch in loader:

            sequence = batch["sequence"].to(device)

            target = batch["target"].squeeze().to(device)

            logits = model.predict(sequence)

            # Mask out items seen in the input (incl. PAD / MASK).
            seen_mask = torch.zeros_like(logits, dtype=torch.bool)
            seen_mask.scatter_(1, sequence, True)

            seen_mask[:, PAD_TOKEN]  = True
            seen_mask[:, MASK_TOKEN] = True

            logits = logits.masked_fill(seen_mask, float('-inf'))

            topk = torch.topk(

                logits,
                k,
                dim=1

            ).indices

            for i in range(target.size(0)):

                ranking = topk[i]

                truth = target[i]

                pos = (

                    ranking == truth

                ).nonzero(as_tuple=True)

                if len(pos[0]) > 0:

                    rank = pos[0].item()

                    score += 1 / math.log2(rank + 2)

                total += 1

    return score / total

In [46]:
print("Hit@5 :", hit_rate(finetune_model, valid_loader, 5))
print("Hit@10:", hit_rate(finetune_model, valid_loader, 10))

print()

print("NDCG@5 :", ndcg(finetune_model, valid_loader, 5))
print("NDCG@10:", ndcg(finetune_model, valid_loader, 10))

Hit@5 : 0.005302226935312832
Hit@10: 0.018027571580063628

NDCG@5 : 0.003006693452931001
NDCG@10: 0.007313982491001439
